<a href="https://colab.research.google.com/github/YunfeiYang1218/rag-10k-final/blob/main/rag_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters faiss-cpu pypdf

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Alphabet 10K 2024.pdf to Alphabet 10K 2024.pdf
Saving MSFT 10-K(1).pdf to MSFT 10-K(1).pdf
Saving Amazon 10K 2024.pdf to Amazon 10K 2024.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

docs = []

files = [
    "Alphabet 10K 2024.pdf",
    "Amazon 10K 2024.pdf",
    "MSFT 10-K(1).pdf"
]

for file in files:
    loader = PyPDFLoader(file)
    docs.extend(loader.load())

print("Total pages loaded:", len(docs))

Total pages loaded: 409


In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter new Gemini API key: ")
print("API key loaded")

Enter new Gemini API key: ··········
API key loaded


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

documents = text_splitter.split_documents(docs)

print("Total chunks:", len(documents))

Total chunks: 868


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

company_docs = {}

file_company_map = {
    "Alphabet 10K 2024.pdf": "Alphabet",
    "Amazon 10K 2024.pdf": "Amazon",
    "MSFT 10-K(1).pdf": "Microsoft"
}

for file, company in file_company_map.items():
    loader = PyPDFLoader(file)
    loaded_docs = loader.load()
    for d in loaded_docs:
        d.metadata["company"] = company
        d.metadata["source_file"] = file
    company_docs[company] = loaded_docs

for company in company_docs:
    print(company, len(company_docs[company]))

Alphabet 99
Amazon 89
Microsoft 221


In [ ]:
import time
from langchain_community.vectorstores import FAISS

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001")

batch_size = 100
vectorstore = None

for i in range(0, len(documents), batch_size):
    batch = documents[i:i+batch_size]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)

print("Vector DB created with all documents")

Vector DB created with all documents


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

print("Retriever ready")

Retriever ready


Group5 Question


In [ ]:
# Group 5: Regulatory Risks Comparison
query = "Using only the information in the latest 10-K filings of Alphabet, Amazon, and Microsoft, identify three regulatory risks they all face."

# Check if required objects exist before running
if 'retriever' not in globals() or 'llm' not in globals():
    print("CRITICAL ERROR: 'retriever' or 'llm' is not defined. Please run the setup cells above first.")
else:
    # 1. Retrieve the relevant document chunks
    docs_found = retriever.invoke(query) #cite: 261

    # 2. Build the context string safely
    context_list = []
    for doc in docs_found:
        # Using .get() to prevent KeyError if metadata is missing
        company = doc.metadata.get('company', 'Unknown Company')
        source = doc.metadata.get('source_file', doc.metadata.get('source', 'Unknown File'))
        page = doc.metadata.get('page', 'N/A')

        header = f"--- Company: {company} | Source: {source} (Page {page}) ---"
        context_list.append(f"{header}\n{doc.page_content}")

    context = "\n\n".join(context_list) #cite: 262-266

    # 3. Define the Prompt
    prompt_text = f"""
    You are a financial analysis assistant.
    Answer the question only based on the provided context from the uploaded 10-K filings.
    When comparing companies, be explicit and name the company.
    Identify three regulatory risks (e.g., AI, Antitrust, Privacy) common to Alphabet, Amazon, and Microsoft.

    Context:
    {context}

    Question:
    {query}
    """

    # 4. Generate response
    print(f"Successfully retrieved {len(docs_found)} document chunks.")
    response = llm.invoke(prompt_text)
    print("\n" + "="*20 + " ANALYSIS RESULT " + "="*20 + "\n")
    print(response.content)

CRITICAL ERROR: 'retriever' or 'llm' is not defined. Please run the setup cells above first.


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

print("Retriever ready")

Retriever ready


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("LLM ready")

LLM ready


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a financial analysis assistant.

Answer the question only based on the provided context.
If the answer is not in the context, say:
"I cannot find the answer in the provided documents."

Context:
{context}

Question:
{input}
""")

print("Prompt ready")

Prompt ready


In [ ]:
query = "What are the main businesses of Amazon?"

docs_found = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:2000])

Retrieved docs: 6
TV, Echo, Ring, Blink, and eero, and we develop and produce media content. We seek to offer our customers low prices, fast and free delivery, easy-to-use
functionality, and timely customer service. In addition, we offer subscription services such as Amazon Prime, a membership program that includes fast, free
shipping on tens of millions of items, access to award-winning movies and series, live sports, and other benefits.
We fulfill customer orders in a number of ways, including through: North America and International fulfillment networks that we operate; co-sourced and
outsourced arrangements in certain countries; digital delivery; and through our physical stores. We operate customer service centers globally, which are
supplemented by co-sourced arrangements. See Item 2 of Part I, “Properties.”
Sellers
We offer programs that enable sellers to grow their businesses, sell their products in our stores, and fulfill orders using our services. We are not the seller
of reco

Group 2 Question

In [ ]:
query = "Which company reported the fastest growth in its cloud segment between 2023 and 2024, and what factors did they attribute this growth to?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
Google subscriptions, platforms, and devices
Google subscriptions, platforms, and devices revenues increased $5.7 billion from 2023 to 2024. The growth was 
primarily driven by an increase in subscription revenues, largely from growth in the number of paid subscribers  for 
YouTube services followed by Google One. 
Google Cloud
Google Cloud revenues increased $10.1 billion from 2023 to 2024 primarily driven by growth in Google Cloud 
Platform largely from infrastructure services.
Revenues by Geography
The following table presents revenues by geography as a percentage of revenues, determined based on the 
addresses of our customers:
 Year Ended December 31,
 2023 2024
United States  47 %  49 %
EMEA  30 %  29 %
APAC  17 %  16 %
Other Americas  6 %  6 %
Hedging gains (losses)  0 %  0 %
For additional information, see Note 2 of the Notes to Consolidated Financial Statements included in Item 8 of this 
Annual Report on Form 10-K.
Use of Non-GAAP Constant Currency Informati

In [ ]:
prompt = f"""
You are a financial analysis assistant.

Answer the question only based on the provided context from the uploaded 10-K filings.
When comparing companies, be explicit and name the company.
If percentages or growth rates are mentioned, include them exactly as stated in the context.
If the answer is not clearly supported by the context, say so instead of guessing.

Context:
{context}

Question:
{query}
"""

In [ ]:
response = llm.invoke(prompt)
print(response.content)

Based on the provided context, only information for Google's cloud segment is available.

Google Cloud revenues increased $10.1 billion from 2023 to 2024. This growth was primarily driven by growth in Google Cloud Platform, largely from infrastructure services.

The context does not provide information for other companies, so it is not possible to determine which company reported the fastest growth in its cloud segment.


Group 1 Question

In [ ]:
query = "Which company appears to have the strongest enterprise AI infrastructure advantage?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
Deliver the Most Advanced, Safe, and Responsible AI
We aim to build the most advanced, safe, and responsible AI through a full stack of robust AI-optimized 
infrastructure, including data centers, chips, and a global fiber network; world class research teams; and a broad global 
reach through products and platforms that touch billions of people and customers around the world.
We are driving efficiencies in our data centers, while making significant hardware and model improvements. For 
example, since we started serving AI Overviews to our users, we have significantly lowered machine costs and latency 
through hardware, engineering, and technical breakthroughs. Our AI-optimized infrastructure allows us to use, and 
offer our customers, a range of AI accelerator options, including our own custom-built Tensor Processing Units (TPUs). 
Our teams across Alphabet leverage Gemini, as well as other AI models we have previously developed and 
announced, to deliver the best pro

Group 3 Question

In [ ]:
query = "According to Amazon’s latest 10-K, what are the main sources of Amazon’s operating income, and which segment contributes the most?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
Table of Contents
General and Administrative
The decrease in general and administrative costs in 2024, compared to the prior year, is primarily due to a decrease in payroll and related expenses.
Other Operating Expense (Income), Net
Other operating expense (income), net was $767 million and $763 million during 2023 and 2024, and was primarily related to asset impairments and the
amortization of intangible assets.
Operating Income (Loss)
Operating income (loss) by segment is as follows (in millions):
Year Ended December 31,
2023 2024
Operating Income (Loss)
North America $ 14,877 $ 24,967 
International (2,656) 3,792 
AWS 24,631 39,834 
Consolidated $ 36,852 $ 68,593 
Operating income was $36.9 billion and $68.6 billion for 2023 and 2024. We believe that operating income is a more meaningful measure than gross
profit and gross margin due to the diversity of our product categories and services. For more information on the operating expenses that impact segment
operating